In [1]:
# ── Imports ───────────────────────────────────────────────────────
from functools import partial
import pybaseball as pb
import pandas as pd
import numpy as np
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
import pyro.infer.autoguide as autoguide
from pyro.optim import Adam
from scipy.stats import spearmanr
from sklearn.preprocessing import RobustScaler

# ── Season Config ─────────────────────────────────────────────────
PRED_SEASON = 2026
TEST_SEASON = PRED_SEASON - 1

DECAY_FEATURES = ['xwOBA', 'BB%', 'K%', 'BsR']
LAMBDA         = 0.5 ** (1/3)

CONT_FEATURES = (
    [f'{f}_decay' for f in ['xwOBA', 'K%', 'BB%', 'BsR']] +
    ['age', 'age_sq', 'experience', 'delta_PA']
)
POSITIONS = ['C', '1B', '2B', '3B', 'SS', 'OF', 'DH', 'UNK']

SEASON_GAMES = {yr: (60 if yr == 2020 else 162) for yr in range(2019, TEST_SEASON + 1)}


# ── Data Pipeline ─────────────────────────────────────────────────
def pull_seasons(seasons):
    return pd.concat(
        [pb.batting_stats(yr, yr, qual=1) for yr in seasons],
        ignore_index=True
    )

def pull_recent_fielding(season):
    return pb.fielding_stats(season, season, qual=0)

def add_fantasy_points(df):
    df = df.copy()
    df['1B'] = df['H'] - df['2B'] - df['3B'] - df['HR']
    df['fantasy_pts'] = (
        df['R']    * 0.75 +
        df['1B']   * 1.0  +
        df['2B']   * 1.5  +
        df['3B']   * 2.0  +
        df['HR']   * 3.0  +
        df['RBI']  * 0.75 +
        df['BB']   * 1.0  +
        df['SO']   * -0.5 +
        df['HBP']  * 1.0  +
        df['SB']   * 1.0  +
        df['CS']   * -2.0 +
        df['GDP']  * -2.0
    )
    df['fantasy_pts_per_PA'] = df['fantasy_pts'] / df['PA'].clip(lower=1)
    return df

def add_primary_pos(df, fld_df):
    primary_pos = (
        fld_df.groupby('IDfg')
        .apply(lambda x: x.loc[x['Inn'].idxmax(), 'Pos'])
        .reset_index(name='Primary_Pos')
    )
    return df.merge(primary_pos, on='IDfg', how='left').fillna({'Primary_Pos': 'UNK'})

def _build_rows(df, include_target=True):
    rows = []
    for pid in df['IDfg'].unique():
        pdf = df[df['IDfg'] == pid].sort_values('Season')
        target_seasons = pdf['Season'].values if include_target else [pdf['Season'].max() + 1]

        for target_season in target_seasons:
            past = pdf[pdf['Season'] < target_season].tail(3)
            if len(past) == 0:
                continue

            w = np.array([LAMBDA ** (len(past) - 1 - j) for j in range(len(past))])
            w /= w.sum()

            # normalize counting stats to per-162 game rate
            past = past.copy()
            season_scale = np.array([162 / SEASON_GAMES.get(s, 162) for s in past['Season'].values])
            past['BsR'] = past['BsR'] * season_scale

            row = {}
            for feat in DECAY_FEATURES:
                vals = past[feat].values.astype(float)
                ww = w.copy()
                ww[np.isnan(vals)] = 0
                row[f'{feat}_decay'] = np.nan if ww.sum() == 0 else (vals * (ww / ww.sum())).sum()

            row['Primary_Pos_recent'] = past['Primary_Pos'].iloc[-1]
            row['age']        = past['Age'].iloc[-1] + 1
            row['age_sq']     = row['age'] ** 2
            row['experience'] = len(past)
            pa_per_162 = past['PA'].values / np.array([SEASON_GAMES.get(s, 162) for s in past['Season'].values]) * 162
            row['delta_PA'] = (pa_per_162[-1] - pa_per_162[-2]) if len(past) >= 2 else 0.0
            # row['delta_PA']   = (past['PA'].iloc[-1] - past['PA'].iloc[-2]
            #                      if len(past) >= 2 else 0.0)

            pos = row['Primary_Pos_recent']
            for p in POSITIONS:
                row[p] = 1.0 if pos == p else 0.0

            row['player'] = pid
            row['season'] = target_season

            if include_target:
                tgt = pdf[pdf['Season'] == target_season].iloc[0]
                sf  = SEASON_GAMES.get(target_season, 162) / 162
                row['target_fantasy_pts_per_PA'] = tgt['fantasy_pts_per_PA']
                row['target_PA']                 = tgt['PA'] * sf

            rows.append(row)
    return pd.DataFrame(rows)

def build_dataset(start_year, end_year):
    df     = pull_seasons(list(range(start_year, end_year + 1)))
    fld_df = pull_recent_fielding(end_year)
    df     = add_fantasy_points(df)
    df     = add_primary_pos(df, fld_df)
    return _build_rows(df, include_target=True)

def build_prediction_dataset(end_year):
    df     = pull_seasons(list(range(end_year - 2, end_year + 1)))
    fld_df = pull_recent_fielding(end_year)
    df     = add_fantasy_points(df)
    df     = add_primary_pos(df, fld_df)
    return _build_rows(df, include_target=False)

def build_dataloaders(df, player_to_idx=None, scaler=None):
    if player_to_idx is None:
        player_to_idx = {pid: idx for idx, pid in enumerate(df['player'].unique())}
    else:
        max_idx = max(player_to_idx.values())
        for pid in df['player'].unique():
            if pid not in player_to_idx:
                max_idx += 1
                player_to_idx[pid] = max_idx

    player_idx = torch.tensor(
        [player_to_idx[pid] for pid in df['player'].values], dtype=torch.long
    )

    # scale continuous features
    X = df[CONT_FEATURES].values
    if scaler is None:
        scaler = RobustScaler()
        X = scaler.fit_transform(X)
    else:
        X = scaler.transform(X)

    data = {
        'player_idx': player_idx,
        'X_cont':     torch.tensor(X, dtype=torch.float),
        'pos_onehot': torch.tensor(df[POSITIONS].values, dtype=torch.float),
        'n_players':  len(player_to_idx),
    }
    if 'target_fantasy_pts_per_PA' in df.columns:
        data['y_pts_per_pa'] = torch.tensor(
            df['target_fantasy_pts_per_PA'].values, dtype=torch.float)
        data['y_pa'] = torch.tensor(
            df['target_PA'].values, dtype=torch.float)

    return data, player_to_idx, scaler

def add_player_names(df):
    additional_lookup = {
        '25477': 'ben rice',
        '29576': 'isaac collins',
        '29518': 'james wood',
        '27464': 'dillon dingler',
        '31781': 'jackson holliday',
        '31595': 'brooks lee',
        '24598': 'addison barger',
        '29794': 'kyle manzardo',
        '24816': 'andy pages',
        '28253': 'Javier Sanoja',
        '23395': 'eric wagaman',
        '31431': 'jordan beck',
        '22857': 'Wenceel Perez',
        '19722': 'Carlos Narvaez',
        '26540': 'Angel Martínez',
        '28312': 'coby mayo',
        '25180': 'daniel schneemann',
        '33541': 'Dylan Crews',
        '31394': 'Brooks Baldwin',
        '26374': 'Kameron Misner',
        '25782': 'Pedro Pagés',
        '29592': 'Connor Norby',
        '29575': 'Trey Sweeney',
        '27883': 'Thomas Saggese',
        '24605': 'Michael Helman',
        '29478': 'Carter Jensen',
        '29995': 'Daylen Lile',
        '25999': 'Victor Mesa Jr.',
        '31812': 'Roman Anthony',
        '31505': 'Sal Stewart',
        '21496': 'Brandon Lockridge',
    }
    ids   = df['player'].unique().tolist()
    names = pb.playerid_reverse_lookup(ids, key_type='fangraphs')[
        ['key_fangraphs', 'name_first', 'name_last']
    ]
    names['name_first'] = names['name_first'].fillna('')
    names['name_last']  = names['name_last'].fillna('')
    names['name'] = (names['name_first'] + ' ' + names['name_last']).str.strip()
    # fallback: use IDfg if name is empty after joining
    names.loc[names['name'] == '', 'name'] = names['key_fangraphs'].astype(str)
    df = df.merge(names[['key_fangraphs', 'name']],
                  left_on='player', right_on='key_fangraphs', how='left')
    # fallback for players not found in lookup at all
    df['name'] = df['name'].fillna(df['player'].astype(str))
    # apply additional_lookup for any remaining missing or ID-only names
    df['name'] = df.apply(
        lambda r: additional_lookup.get(str(int(r['player'])), r['name'])
        if r['name'] == str(r['player']) or r['name'] == '' else r['name'],
        axis=1
    )
    return df


# ── Models ────────────────────────────────────────────────────────
def model_pts_per_pa(player_idx, X_cont, y=None, n_players=500):
    n_cont = X_cont.shape[1]

    alpha   = pyro.sample('alpha',   dist.Normal(0.5, 1.0))
    beta    = pyro.sample('beta',    dist.StudentT(3., torch.zeros(n_cont), torch.ones(n_cont)).to_event(1))
    sigma   = pyro.sample('sigma',   dist.HalfNormal(0.2))
    sigma_u = pyro.sample('sigma_u', dist.HalfNormal(0.2))

    with pyro.plate('players', n_players):
        u_players = pyro.sample('player_effect', dist.Normal(0., sigma_u))

    with pyro.plate('obs', X_cont.shape[0]):
        safe_idx = player_idx.clamp(0, n_players - 1)
        u = u_players[safe_idx] * (player_idx < n_players).float()
        mu = alpha + u + (X_cont * beta).sum(-1)
        pyro.sample('y', dist.Normal(mu, sigma), obs=y)

def model_pa(player_idx, X_cont, pos_onehot, y=None, n_players=500):
    n_cont = X_cont.shape[1]

    alpha    = pyro.sample('alpha',    dist.Normal(torch.log(torch.tensor(650.)), 0.5))
    beta     = pyro.sample('beta',     dist.StudentT(3., torch.zeros(n_cont),              torch.ones(n_cont)*0.3).to_event(1))
    beta_pos = pyro.sample('beta_pos', dist.StudentT(3., torch.zeros(pos_onehot.shape[1]), torch.ones(pos_onehot.shape[1])*0.3).to_event(1))
    sigma    = pyro.sample('sigma',    dist.HalfNormal(0.1))
    sigma_u  = pyro.sample('sigma_u',  dist.HalfNormal(0.2))

    with pyro.plate('players', n_players):
        u_players = pyro.sample('player_effect', dist.Normal(0., sigma_u))

    with pyro.plate('obs', X_cont.shape[0]):
        safe_idx = player_idx.clamp(0, n_players - 1)
        u = u_players[safe_idx] * (player_idx < n_players).float()
        mu = alpha + u + (pos_onehot * beta_pos).sum(-1) + (X_cont * beta).sum(-1)
        pyro.sample('y_log', dist.Normal(mu, sigma),
                    obs=y.log() if y is not None else None)


# ── Training ──────────────────────────────────────────────────────
def train_model(model_fixed, args_fn, data, n_steps=1000, lr=0.01, n_samples=500):
    pyro.clear_param_store()
    guide = autoguide.AutoDiagonalNormal(model_fixed)
    svi = SVI(model_fixed, guide, Adam({'lr': lr}), Trace_ELBO(num_particles=4))
    losses = []
    for step in range(n_steps):
        loss = svi.step(*args_fn(data))
        losses.append(loss)
        if step % 200 == 0:
            print(f'  step {step:4d}  loss={loss:.1f}')
    print(f'  final loss={losses[-1]:.1f}, initial loss={losses[0]:.1f}')
    # draw samples while param store is clean
    posterior_samples = Predictive(guide, num_samples=n_samples)(*args_fn(data))

    # diagnostic: print posterior medians
    medians = guide.median()
    for k, v in medians.items():
        print(f'  {k}: {v.detach().flatten()[:5].numpy()}')  # first 5 values

    # exclude auxiliary latent and squeeze extra dim
    exclude = {'_AutoDiagonalNormal_latent'}
    posterior_samples = {
        k: v.squeeze(1) 
        for k, v in posterior_samples.items() 
        if k not in exclude
    }
    print("posterior keys:", list(posterior_samples.keys()))
    print("sample shapes:", {k: v.shape for k, v in posterior_samples.items()})
    return posterior_samples

def train(data):
    n_players = data['n_players']
    model_pts_fixed = partial(model_pts_per_pa, n_players=n_players)
    model_pa_fixed  = partial(model_pa,         n_players=n_players)

    print('Training pts_per_pa...')
    pts_samples = train_model(
        model_pts_fixed,
        lambda d: (d['player_idx'], d['X_cont'], d['y_pts_per_pa']),
        data, n_steps=3000, lr=0.01
    )

    print('Training pa...')
    pa_samples = train_model(
        model_pa_fixed,
        lambda d: (d['player_idx'], d['X_cont'], d['pos_onehot'], d['y_pa']),
        data, n_steps=2000, lr=0.01
    )

    return pts_samples, pa_samples, model_pts_fixed, model_pa_fixed

def predict(pts_samples, pa_samples, model_pts_fixed, model_pa_fixed, data, n_samples=500):
    pred_pts = Predictive(model_pts_fixed, posterior_samples=pts_samples, num_samples=n_samples)(
        data['player_idx'], data['X_cont']
    )['y']

    pred_pa = Predictive(model_pa_fixed, posterior_samples=pa_samples, num_samples=n_samples)(
        data['player_idx'], data['X_cont'], data['pos_onehot']
    )['y_log'].clamp(max=torch.log(torch.tensor(750.))).exp()

    tot = pred_pts * pred_pa  # [n_samples, n_players]

    return {
        'mean_pts':     tot.mean(0).detach().numpy(),
        'std_pts':      tot.std(0).detach().numpy(),
        'mean_pts_per_pa': pred_pts.mean(0).detach().numpy(),
        'mean_pa':      pred_pa.mean(0).detach().numpy(),
    }

def risk_adjusted_score(mean, std, alpha=0.5):
    return mean / (std ** alpha)


# ── Evaluation ────────────────────────────────────────────────────
def evaluate(data):
    train_raw = data[data['season'] < TEST_SEASON].copy()
    test_raw  = data[data['season'] == TEST_SEASON].copy().reset_index(drop=True)

    player_to_idx = {pid: idx for idx, pid in enumerate(train_raw['player'].unique())}
    train_data, player_to_idx, scaler = build_dataloaders(train_raw, player_to_idx)
    test_data,  _, _                  = build_dataloaders(test_raw, player_to_idx, scaler=scaler)

    pts_samples, pa_samples, model_pts_fixed, model_pa_fixed = train(train_data)
    # pred_tot, pred_pts_per_pa, pred_pa = predict(
    #     pts_samples, pa_samples, model_pts_fixed, model_pa_fixed, test_data
    # )

    preds = predict(pts_samples, pa_samples, model_pts_fixed, model_pa_fixed, test_data)
    pred_tot = preds['mean_pts']
    pred_pts_per_pa = preds['mean_pts_per_pa']
    pred_pa = preds['mean_pa']

    # attach predictions to test_raw
    test_raw['pred_fantasy_pts'] = pred_tot
    test_raw['pred_pts_per_pa']  = pred_pts_per_pa
    test_raw['pred_pa']          = pred_pa
    test_raw['true_fantasy_pts'] = test_raw['target_fantasy_pts_per_PA'] * test_raw['target_PA']

    # add player names
    ids = test_raw['player'].unique().tolist()
    names = pb.playerid_reverse_lookup(ids, key_type='fangraphs')[['key_fangraphs', 'name_first', 'name_last']]
    names['name'] = names['name_first'] + ' ' + names['name_last']
    test_raw = test_raw.merge(names[['key_fangraphs', 'name']], 
                               left_on='player', right_on='key_fangraphs', how='left')

    corr_tot,        _ = spearmanr(pred_tot, test_raw['true_fantasy_pts'])
    corr_pts_per_pa, _ = spearmanr(pred_pts_per_pa, test_raw['target_fantasy_pts_per_PA'])
    corr_pa,         _ = spearmanr(pred_pa, test_raw['target_PA'])

    print(f'Total pts  Spearman: {corr_tot:.4f}')
    print(f'pts_per_PA Spearman: {corr_pts_per_pa:.4f}')
    print(f'PA         Spearman: {corr_pa:.4f}')

    return {
        'corr_tot':        corr_tot,
        'corr_pts_per_pa': corr_pts_per_pa,
        'corr_pa':         corr_pa,
        'results':         test_raw,  # full DataFrame with predictions + names
        'pts_samples':     pts_samples,
        'pa_samples':      pa_samples,
        'model_pts_fixed': model_pts_fixed,
        'model_pa_fixed':  model_pa_fixed,
        'player_to_idx':   player_to_idx,
        'scaler':          scaler,
    }


# ── Draft Rankings ────────────────────────────────────────────────
def get_draft_rankings(end_year=TEST_SEASON, alpha=0.5):
    pred_df  = build_prediction_dataset(end_year).dropna(subset=CONT_FEATURES)
    full_raw = build_dataset(2019, end_year).dropna(subset=CONT_FEATURES)

    player_to_idx = {pid: idx for idx, pid in enumerate(full_raw['player'].unique())}
    full_data, player_to_idx, scaler = build_dataloaders(full_raw, player_to_idx)
    pred_data, _, _                  = build_dataloaders(pred_df, player_to_idx, scaler=scaler)

    pts_samples, pa_samples, model_pts_fixed, model_pa_fixed = train(full_data)
    preds = predict(pts_samples, pa_samples, model_pts_fixed, model_pa_fixed, pred_data)

    # add player names
    pred_df = add_player_names(pred_df)
    # ids   = pred_df['player'].unique().tolist()
    # names = pb.playerid_reverse_lookup(ids, key_type='fangraphs')[['key_fangraphs', 'name_first', 'name_last']]
    # names['name'] = names['name_first'] + ' ' + names['name_last']

    pred_df = pred_df.copy()
    pred_df['mean_pts']   = preds['mean_pts']
    pred_df['std_pts']    = preds['std_pts']
    pred_df['mean_pts_per_pa'] = preds['mean_pts_per_pa']
    pred_df['pred_pa']    = preds['mean_pa']
    pred_df['risk_score'] = risk_adjusted_score(preds['mean_pts'], preds['std_pts'], alpha=alpha)
    # pred_df = pred_df.merge(names[['key_fangraphs', 'name']],
    #                          left_on='player', right_on='key_fangraphs', how='left')
    RANKING_COLS = [
        'name', 'player', 'Primary_Pos_recent',
        'mean_pts', 'std_pts', 'risk_score',
        'mean_pts_per_pa', 'pred_pa',
        'age', 'experience',
        'xwOBA_decay', 'BB%_decay', 'K%_decay', 'BsR_decay',
    ]

    return (pred_df
            .sort_values('risk_score', ascending=False)
            .reset_index(drop=True)
            [RANKING_COLS]
    )



# ── Entry Point ───────────────────────────────────────────────────
# if __name__ == '__main__':
#     print('Building dataset...')
#     data = build_dataset(2019, TEST_SEASON)
#     data = data.dropna(subset=CONT_FEATURES)

    # print('Evaluating...')
    # results = evaluate(data)

/opt/miniconda3/envs/dugout/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rankings = get_draft_rankings(alpha = 0.05)
rankings.to_csv('./outputs/hitter_rankings.csv', index=False)


/var/folders/_y/9fww9bxx4qlc9wkrxyt0_6h40000gn/T/ipykernel_55280/1824292025.py:64: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.loc[x['Inn'].idxmax(), 'Pos'])
/var/folders/_y/9fww9bxx4qlc9wkrxyt0_6h40000gn/T/ipykernel_55280/1824292025.py:64: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.loc[x['Inn'].idxmax(), 'Pos'])


Training pts_per_pa...
  step    0  loss=277380.9
  step  200  loss=1512.2
  step  400  loss=623.2
  step  600  loss=139.4
  step  800  loss=115.2
  step 1000  loss=-144.6
  step 1200  loss=-311.9
  step 1400  loss=-320.5
  step 1600  loss=-371.2
  step 1800  loss=-357.7
  step 2000  loss=-416.4
  step 2200  loss=-423.3
  step 2400  loss=-472.1
  step 2600  loss=-454.9
  step 2800  loss=-445.4
  final loss=-443.9, initial loss=277380.9
  alpha: [0.33441174]
  beta: [ 0.04042294 -0.02289744  0.00278125  0.0044516  -0.03881542]
  sigma: [0.16708763]
  sigma_u: [0.1772758]
  player_effect: [0.12449724 0.09685644 0.18743262 0.07801956 0.13546677]
posterior keys: ['alpha', 'beta', 'sigma', 'sigma_u', 'player_effect']
sample shapes: {'alpha': torch.Size([500]), 'beta': torch.Size([500, 8]), 'sigma': torch.Size([500]), 'sigma_u': torch.Size([500]), 'player_effect': torch.Size([500, 1212])}
Training pa...
  step    0  loss=1793666.1
  step  200  loss=92321.8
  step  400  loss=46245.1
  step  6